In [1]:
import math
import warnings
from pathlib import Path

import rasterio
import numpy as np
from pathlib import Path

import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio import features
from shapely import wkb


In [18]:
def tif_info(path, sample_pixels=250_000):

    path = Path(path)

    with rasterio.open(path) as ds:
        width, height = ds.width, ds.height
        bands = ds.count
        dtype = ds.dtypes[0]

        # ---- sampling window ----
        side = int(np.sqrt(sample_pixels))
        w = min(width, side)
        h = min(height, side)

        data = ds.read(1, window=((0, h), (0, w))).astype("float64")

        # ---- NaNs ----
        nan_mask = np.isnan(data)
        nan_count = int(nan_mask.sum())
        nan_fraction = nan_count / data.size

        valid = data[~nan_mask]
        if valid.size == 0:
            return {
                "file": path.name,
                "error": "all sampled values are NaN",
            }

        # ---- stats ----
        unique_vals = np.unique(valid)

        # categorical vs continuous heuristic
        if dtype.startswith(("int", "uint")) and unique_vals.size < 256:
            value_type = "categorical (very likely)"
        elif dtype.startswith(("int", "uint")):
            value_type = "integer (uncertain)"
        else:
            value_type = "continuous"

        return {
            "file": path.name,
            "dimensions": (height, width),
            "bands": bands,
            "dtype": dtype,
            "sample_shape": (h, w),
            "nan_count": nan_count,
            "nan_fraction": nan_fraction,
            "min": float(valid.min()),
            "max": float(valid.max()),
            "mean": float(valid.mean()),
            "std": float(valid.std()),
            "p50": float(np.percentile(valid, 50)),
            "p90": float(np.percentile(valid, 90)),
            "p95": float(np.percentile(valid, 95)),
            "p99": float(np.percentile(valid, 99)),
            "frac_zero": float((valid == 0).mean()),
            "unique_values_sampled": int(unique_vals.size),
            "value_type": value_type,
        }




In [19]:
def print_stats(stats):
    print("=" * 70)
    print(f"File: {stats.get('file', 'unknown')}")
    print("=" * 70)

    def fmt(v):
        if isinstance(v, float):
            return f"{v:.6g}"
        return v

    order = [
        "dimensions",
        "bands",
        "dtype",
        "value_type",
        "sample_shape",
        "nan_count",
        "nan_fraction",
        "min",
        "max",
        "mean",
        "std",
        "p50",
        "p90",
        "p95",
        "p99",
        "frac_zero",
        "unique_values_sampled",
    ]

    for key in order:
        if key in stats:
            print(f"{key:<22}: {fmt(stats[key])}")


In [21]:
gdp_info = tif_info("READY_data\inputs/2019_gdp_aligned_025deg.tif")
pop_info = tif_info("READY_data\inputs/2020_pop_aligned_025deg.tif")
landcover_info = tif_info("READY_data\inputs/2018_landcover_aligned_025deg.tif")
historic_3band_25deg = tif_info("READY_data\inputs/historic_3band_025deg.tif")

print_stats(gdp_info)
print_stats(pop_info)
print_stats(landcover_info)
print_stats(historic_3band_25deg)

File: 2019_gdp_aligned_025deg.tif
dimensions            : (195, 228)
bands                 : 1
dtype                 : float32
value_type            : continuous
sample_shape          : (195, 228)
nan_count             : 0
nan_fraction          : 0
min                   : 0
max                   : 101.111
mean                  : 1.2186
std                   : 4.68893
p50                   : 0
p90                   : 2.46808
p95                   : 6.2408
p99                   : 23.9419
frac_zero             : 0.639879
unique_values_sampled : 15107
File: 2020_pop_aligned_025deg.tif
dimensions            : (195, 228)
bands                 : 1
dtype                 : float64
value_type            : continuous
sample_shape          : (195, 228)
nan_count             : 0
nan_fraction          : 0
min                   : 0
max                   : 7.23456e+06
mean                  : 16541.9
std                   : 95020.3
p50                   : 0
p90                   : 34177.4
p95          

In [1]:
# Restructure the historic lc dataset to be 7 layered one-hot-encoded
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

# --- user inputs ---
tif_path = Path("READY_data\inputs/2018_landcover_aligned_025deg.tif")  # <-- change if needed
nodata_override = None  # e.g. 255 or -128 if your file uses a special nodata; else leave None

# --- load raster ---
with rasterio.open(tif_path) as src:
    arr = src.read(1)  # single band
    nodata = src.nodata if nodata_override is None else nodata_override
    profile = src.profile

print("File:", tif_path)
print("Shape (H, W):", arr.shape)
print("Dtype:", arr.dtype)
print("Raster nodata in metadata:", src.nodata)

# --- mask nodata if present ---
mask = np.ones(arr.shape, dtype=bool)
if nodata is not None:
    mask &= (arr != nodata)

valid = arr[mask]

# --- uniques + counts ---
vals, counts = np.unique(valid, return_counts=True)
total = counts.sum()
fractions = counts / total

# --- build summary table ---
df = pd.DataFrame({
    "class_id": vals.astype(int),
    "count": counts.astype(int),
    "fraction": fractions,
})
df["percent"] = df["fraction"] * 100

# sort by class_id (use sort by count if you prefer)
df = df.sort_values("class_id").reset_index(drop=True)

print("\nUnique class IDs (valid pixels):", len(df))
print("Min/Max class ID:", int(df["class_id"].min()), int(df["class_id"].max()))
print("\nTop 20 classes by pixel count:")
display(df.sort_values("count", ascending=False).head(20))

print("\nAll classes (sorted by class_id):")
display(df)

# --- quick checks ---
missing_ids = set(range(int(df["class_id"].min()), int(df["class_id"].max()) + 1)) - set(df["class_id"].tolist())
print("\nMissing IDs within min..max (if any):", sorted(missing_ids)[:50], ("..." if len(missing_ids) > 50 else ""))

# --- save for manual crosswalk work ---
out_csv = tif_path.with_suffix("").as_posix() + "_classes.csv"
df.to_csv(out_csv, index=False)
print("\nSaved class frequency table to:", out_csv)


File: READY_data\inputs\2018_landcover_aligned_025deg.tif
Shape (H, W): (195, 228)
Dtype: int8
Raster nodata in metadata: 0.0

Unique class IDs (valid pixels): 44
Min/Max class ID: 1 44

Top 20 classes by pixel count:


,class_id,count,fraction,percent
43,44,2929,0.205299,20.529894
11,12,2233,0.156515,15.651503
23,24,1939,0.135908,13.590804
22,23,1138,0.079764,7.976449
17,18,855,0.059929,5.992851
24,25,635,0.044508,4.450831
28,29,515,0.036097,3.609729
26,27,473,0.033153,3.315343
19,20,438,0.030700,3.070022
20,21,425,0.029789,2.978902



All classes (sorted by class_id):


,class_id,count,fraction,percent
0,1,17,0.001192,0.119156
1,2,307,0.021518,2.151819
2,3,69,0.004836,0.483634
3,4,8,0.000561,0.056073
4,5,1,0.000070,0.007009
5,6,10,0.000701,0.070092
6,7,19,0.001332,0.133174
7,8,1,0.000070,0.007009
8,9,7,0.000491,0.049064
9,10,9,0.000631,0.063083



Missing IDs within min..max (if any): [] 

Saved class frequency table to: READY_data/inputs/2018_landcover_aligned_025deg_classes.csv


In [3]:
with rasterio.open("READY_data\inputs/2018_landcover_aligned_025deg.tif") as src:
    print("Tags:")
    for k, v in src.tags().items():
        print(f"{k}: {v}")


Tags:
AREA_OR_POINT: Area


In [4]:
with rasterio.open("READY_data\inputs/2018_landcover_aligned_025deg.tif") as src:
    print("Band 1 tags:")
    print(src.tags(1))


Band 1 tags:
{'STATISTICS_MAXIMUM': '44', 'STATISTICS_MEAN': '26.215251980094', 'STATISTICS_MINIMUM': '1', 'STATISTICS_STDDEV': '11.618476357186', 'STATISTICS_VALID_PERCENT': '32.09'}
